# Agent Tooling Demo

**The Tooling Layer of the AI Agent Stack**

This notebook demonstrates the core capabilities of `agent-tooling`:

1. **Tool Definition** - Using the `@tool` decorator
2. **Built-in Tools** - Developer, Data, and Cognitive tools
3. **Tool Composition** - Building complex workflows
4. **MCP Integration** - Model Context Protocol
5. **Arena Mode** - Comparing tools side-by-side
6. **Schema Generation** - OpenAI function calling format

---

## Setup

First, let's install and import the package.

In [ ]:
# Install if needed
# !pip install agent-tooling

# Or install from local source
# !pip install -e ..

In [ ]:
# Core imports
from agent_tooling import tool, ToolRegistry, ToolResult, ToolError
from agent_tooling.interceptor import ToolingInterceptor
from agent_tooling.composer import ToolComposer, ToolStep, CompositeTool

# Built-in tools
from agent_tooling.tools.developer import read_file, write_file, list_directory, file_exists, execute_python
from agent_tooling.tools.data import query_database, call_api, fetch_json, scrape_webpage
from agent_tooling.tools.cognitive import calculate, web_search, wikipedia_search

# Visualization
from agent_tooling.visualization import ToolArena, ToolTracer

import json
print("agent-tooling loaded successfully!")
print(f"Registered tools: {ToolRegistry.count()}")

---

## 1. Tool Definition with `@tool` Decorator

The `@tool` decorator is the primary way to define tools. It automatically:
- Extracts parameters from the function signature
- Parses docstrings for descriptions
- Generates JSON schemas for function calling
- Registers the tool in the global registry

In [ ]:
# Define a custom tool
@tool(name="greeting_generator", category="demo", mcp_enabled=True)
def greeting_generator(name: str, language: str = "english", formal: bool = False) -> str:
    """Generate a greeting message in different languages.
    
    Args:
        name: Name of the person to greet
        language: Language for the greeting (english, spanish, french, japanese)
        formal: Whether to use formal language
    
    Returns:
        A greeting message in the specified language
    """
    greetings = {
        "english": ("Hello", "Good day"),
        "spanish": ("Hola", "Buenos días"),
        "french": ("Salut", "Bonjour"),
        "japanese": ("やあ", "こんにちは"),
    }
    
    informal, formal_greeting = greetings.get(language.lower(), greetings["english"])
    greeting = formal_greeting if formal else informal
    
    return f"{greeting}, {name}!"

# Test the tool
result = greeting_generator(name="Alice", language="japanese", formal=True)
print(f"Result: {result.data}")
print(f"Success: {result.success}")
print(f"Execution time: {result.execution_time_ms:.2f}ms")

In [ ]:
# View the generated schema
print("OpenAI Function Schema:")
print(json.dumps(greeting_generator.to_openai_function(), indent=2))

In [ ]:
# View MCP tool format
print("MCP Tool Schema:")
print(json.dumps(greeting_generator.to_mcp_tool(), indent=2))

---

## 2. Built-in Tools

### 2.1 Cognitive Tools

Tools that enhance agent reasoning capabilities.

In [ ]:
# Calculator - safe mathematical expression evaluation
expressions = [
    "2 + 2",
    "sqrt(144) + pow(2, 8)",
    "sin(pi/2) * 100",
    "log10(1000) + log2(8)",
]

print("Calculator Tool Demo:")
print("=" * 50)
for expr in expressions:
    result = calculate(expression=expr)
    print(f"{expr:30} = {result.data['result']}")

In [ ]:
# Wikipedia search
result = wikipedia_search(query="artificial intelligence", sentences=3)
print("Wikipedia Search Result:")
print("=" * 50)
if result.success and result.data.get('found'):
    print(f"Title: {result.data['title']}")
    print(f"URL: {result.data['url']}")
    print(f"\nSummary:\n{result.data['summary']}")
else:
    print("No results found")

### 2.2 Developer Tools

Tools for file system and code execution.

In [ ]:
# List directory contents
result = list_directory(path="..", pattern="*.py")
print("Python files in parent directory:")
print("=" * 50)
for item in result.data[:5]:
    print(f"  {item['name']}")

In [ ]:
# Execute Python code
code = """
import math

def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# Calculate first 10 fibonacci numbers
result = [fibonacci(i) for i in range(10)]
print(f"Fibonacci sequence: {result}")
"""

result = execute_python(code=code)
print("Python Execution Result:")
print("=" * 50)
print(f"Success: {result.data['success']}")
print(f"Output: {result.data['stdout']}")

### 2.3 Data Tools

Tools for database and API access.

In [ ]:
# Fetch JSON from an API
result = fetch_json(url="https://api.github.com/repos/anthropics/anthropic-cookbook")
print("GitHub API Result:")
print("=" * 50)
if result.success:
    data = result.data
    print(f"Repo: {data.get('full_name')}")
    print(f"Stars: {data.get('stargazers_count')}")
    print(f"Description: {data.get('description', 'N/A')[:80]}...")
else:
    print(f"Error: {result.error}")

In [ ]:
# Database query (mock mode - no real connection)
result = query_database(sql="SELECT * FROM users WHERE active = true LIMIT 5")
print("Database Query Result (Mock):")
print("=" * 50)
print(json.dumps(result.data, indent=2))

---

## 3. Tool Composition

Build complex workflows by composing multiple tools together.

In [ ]:
# First, let's create some atomic tools for our composition demo

@tool(name="fetch_data", category="demo")
def fetch_data(source: str) -> dict:
    """Fetch data from a source.
    
    Args:
        source: The data source identifier
    """
    # Simulated data fetch
    return {
        "source": source,
        "records": [
            {"id": 1, "value": 100},
            {"id": 2, "value": 200},
            {"id": 3, "value": 300},
        ]
    }

@tool(name="process_data", category="demo")
def process_data(records: list) -> dict:
    """Process a list of records.
    
    Args:
        records: List of data records to process
    """
    total = sum(r.get("value", 0) for r in records)
    avg = total / len(records) if records else 0
    return {
        "total": total,
        "average": avg,
        "count": len(records)
    }

@tool(name="format_report", category="demo")
def format_report(stats: dict, title: str = "Report") -> str:
    """Format statistics into a report.
    
    Args:
        stats: Statistics dictionary
        title: Report title
    """
    return f"""
=== {title} ===
Total: {stats.get('total', 'N/A')}
Average: {stats.get('average', 'N/A'):.2f}
Count: {stats.get('count', 'N/A')}
"""

print("Atomic tools created!")

In [ ]:
# Create a composite tool using ToolComposer
from agent_tooling.composer import CompositionMode

data_pipeline = CompositeTool(
    name="data_analysis_pipeline",
    description="Fetch, process, and report on data",
    steps=[
        ToolStep(
            tool_name="fetch_data",
            parameters={"source": "sales_db"},
            output_key="raw_data"
        ),
        ToolStep(
            tool_name="process_data",
            input_mapping={"records": "raw_data"},  # Map from previous output
            output_key="stats"
        ),
        ToolStep(
            tool_name="format_report",
            input_mapping={"stats": "stats"},
            parameters={"title": "Sales Analysis"}
        ),
    ],
    mode=CompositionMode.SEQUENTIAL
)

# Note: We need to fix the input mapping for the composite tool
# For now, let's run the tools manually to demonstrate
print("Running data pipeline manually:")
print("=" * 50)

# Step 1: Fetch
fetch_result = fetch_data(source="sales_db")
print(f"1. Fetched {len(fetch_result.data['records'])} records")

# Step 2: Process
process_result = process_data(records=fetch_result.data['records'])
print(f"2. Processed: total={process_result.data['total']}, avg={process_result.data['average']}")

# Step 3: Format
report_result = format_report(stats=process_result.data, title="Sales Analysis")
print(f"3. Generated report:")
print(report_result.data)

---

## 4. Tool Registry

The registry tracks all available tools and provides schema generation.

In [ ]:
# List all registered tools by category
print("Registered Tools by Category:")
print("=" * 50)

for category in ToolRegistry.get_categories():
    tools = ToolRegistry.get_by_category(category)
    print(f"\n{category.upper()} ({len(tools)} tools):")
    for tool in tools:
        print(f"  - {tool.name}: {tool.description[:50]}...")

In [ ]:
# Generate OpenAI function schemas for all tools
schemas = ToolRegistry.to_openai_functions()
print(f"Generated {len(schemas)} OpenAI function schemas")
print("\nExample schema (calculate):")
calc_schema = next(s for s in schemas if s['name'] == 'calculate')
print(json.dumps(calc_schema, indent=2))

---

## 5. Tool Interceptor

The interceptor routes tool calls and handles different input formats.

In [ ]:
# Create an interceptor
interceptor = ToolingInterceptor(verbose=False)

# Execute tools by name
result = interceptor.execute("calculate", expression="2 ** 10")
print(f"Direct execution: 2^10 = {result.data['result']}")

# Execute from OpenAI function call format
function_call = {
    "name": "wikipedia_search",
    "arguments": json.dumps({"query": "Python programming", "sentences": 2})
}
result = interceptor.execute_function_call(function_call)
print(f"\nFunction call execution:")
if result.success and result.data.get('found'):
    print(f"  Title: {result.data['title']}")
    print(f"  Summary: {result.data['summary'][:100]}...")

In [ ]:
# View execution trace
print("Execution Trace:")
print("=" * 50)
for entry in interceptor.trace:
    status = "✓" if entry['success'] else "✗"
    print(f"{status} {entry['tool']}: {entry['execution_time_ms']:.2f}ms")

---

## 6. Arena Mode - Tool Comparison

Compare multiple tools on the same task.

In [ ]:
# Create tools for comparison
@tool(name="search_v1", category="arena_demo")
def search_v1(query: str) -> str:
    """Search implementation v1 (simple).
    
    Args:
        query: Search query
    """
    import time
    time.sleep(0.1)  # Simulate work
    return f"V1 results for: {query}"

@tool(name="search_v2", category="arena_demo")
def search_v2(query: str) -> str:
    """Search implementation v2 (optimized).
    
    Args:
        query: Search query
    """
    import time
    time.sleep(0.05)  # Faster!
    return f"V2 results for: {query}"

@tool(name="search_v3", category="arena_demo")
def search_v3(query: str) -> str:
    """Search implementation v3 (experimental).
    
    Args:
        query: Search query
    """
    import time
    time.sleep(0.02)  # Fastest!
    return f"V3 results for: {query}"

print("Arena demo tools created!")

In [ ]:
# Run arena comparison
arena = ToolArena(verbose=True)
results = arena.compare(
    tools=["search_v1", "search_v2", "search_v3"],
    query="machine learning tutorials"
)

---

## 7. Execution Tracing

Visualize step-by-step tool execution.

In [ ]:
# Create a tracer
tracer = ToolTracer(verbose=True)

# Trace a workflow
with tracer.trace("data_analysis") as trace:
    trace.add_step("load_config", "input", {"config_file": "settings.yaml"})
    
    # Simulate fetching data
    result = fetch_data(source="analytics")
    trace.add_step("fetch_data", "process", {"records": len(result.data['records'])})
    
    # Process
    stats = process_data(records=result.data['records'])
    trace.add_step("compute_stats", "process", stats.data)
    
    # Set final result
    trace.result = stats.data

---

## 8. MCP Schema Generation

Generate schemas for Model Context Protocol integration.

In [ ]:
# Get MCP-enabled tools
mcp_tools = ToolRegistry.get_mcp_tools()
print(f"MCP-enabled tools: {len(mcp_tools)}")

# Generate MCP tool list
mcp_schemas = ToolRegistry.to_mcp_tools()
print(f"\nMCP Tool Schemas ({len(mcp_schemas)} tools):")
print("=" * 50)

# Show first 3
for schema in mcp_schemas[:3]:
    print(f"\n{schema['name']}:")
    print(f"  Description: {schema['description'][:60]}...")
    print(f"  Parameters: {list(schema['inputSchema'].get('properties', {}).keys())}")

---

## 9. Demo: Research Assistant Workflow

A practical example combining multiple tools for a research task.

In [ ]:
def research_assistant(topic: str):
    """Research a topic using multiple tools."""
    print(f"Researching: {topic}")
    print("=" * 60)
    
    # Step 1: Get Wikipedia summary
    print("\n1. Fetching Wikipedia summary...")
    wiki_result = wikipedia_search(query=topic, sentences=3)
    if wiki_result.success and wiki_result.data.get('found'):
        print(f"   Title: {wiki_result.data['title']}")
        print(f"   Summary: {wiki_result.data['summary'][:200]}...")
    
    # Step 2: Calculate something related
    print("\n2. Performing calculations...")
    calc_result = calculate(expression="log2(1024) * 10")
    print(f"   log2(1024) * 10 = {calc_result.data['result']}")
    
    # Step 3: Check if we have local notes
    print("\n3. Checking for local notes...")
    notes_path = f"/tmp/research_{topic.replace(' ', '_')}.txt"
    exists_result = file_exists(path=notes_path)
    print(f"   Notes file exists: {exists_result.data['exists']}")
    
    # Step 4: Save research summary
    print("\n4. Saving research summary...")
    summary = f"""Research Summary: {topic}
{'=' * 40}

Wikipedia: {wiki_result.data.get('summary', 'N/A')[:300] if wiki_result.success else 'Not found'}

Calculations: {calc_result.data['result']}

Generated by agent-tooling
"""
    save_result = write_file(path=notes_path, content=summary)
    print(f"   Saved to: {save_result.data['path']}")
    print(f"   Bytes written: {save_result.data['bytes_written']}")
    
    print("\n" + "=" * 60)
    print("Research complete!")
    return notes_path

# Run the research assistant
output_file = research_assistant("machine learning")

In [ ]:
# Read back the saved research
result = read_file(path=output_file)
print("Saved Research Summary:")
print(result.data)

---

## Summary

This notebook demonstrated the core capabilities of `agent-tooling`:

1. **`@tool` Decorator**: Easy tool definition with automatic schema generation
2. **Built-in Tools**: Developer, Data, and Cognitive categories
3. **Tool Composition**: Building pipelines from atomic tools
4. **Registry**: Central tool management and discovery
5. **Interceptor**: Unified tool execution interface
6. **Arena Mode**: Compare tool performance
7. **Tracing**: Step-by-step execution visualization
8. **MCP Integration**: Model Context Protocol support

### Next Steps

- Explore the CLI: `agent-tooling --help`
- Start the HTTP server: `agent-tooling-server`
- Run MCP server: `agent-tooling --mcp`
- Check out [agent-reasoning](https://github.com/jasperan/agent-reasoning) for the Reasoning Layer